# ⚾ MLB Daily Prediction Tool
### Strikeout O/U + NRFI/YRFI — Run Every Morning Before First Pitch

---

## Two models, one notebook

| Model | Backtest Accuracy | Use as |
|---|---|---|
| **Strikeout O/U** | **67.6%** | Primary — bet STRONG and LEAN calls |
| **NRFI/YRFI** | **52-56%** | Filter — only bet high-conviction calls |

## Reading the output
- **STRONG** = edge > 1.5 Ks (K model) or > 7% (NRFI) — highest confidence
- **LEAN** = edge 0.75-1.5 Ks (K model) or 3-7% (NRFI) — good confidence
- **SKIP** = toss-up — do not bet

**Runtime → Run All. Everything else is automatic.**

## 1 · Setup — Google Drive + Imports

Mounts Google Drive to persist the Statcast cache between sessions.

In [ ]:
# ── GOOGLE DRIVE SETUP ───────────────────────────────────────────────────
# Mounts your Google Drive so the Statcast cache persists between sessions.
# First run: downloads ~2.4GB and saves to Drive (~20 min).
# Every run after: loads from Drive (~2 min).
#
# When prompted, click the link, sign in to your Google account, copy the
# code it gives you, and paste it into the box that appears here.

import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/mlb_cache/cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    os.chdir('/content/drive/MyDrive/mlb_cache')
    print(f'Google Drive mounted. Cache: {CACHE_DIR}')
    print(f'Existing cache files: {os.listdir(CACHE_DIR) if os.path.exists(CACHE_DIR) else "none yet"}')
except ImportError:
    # Running locally — use local cache folder
    CACHE_DIR = 'cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    print(f'Running locally. Cache: {CACHE_DIR}')
    print(f'Existing cache files: {os.listdir(CACHE_DIR)}')

In [ ]:
%%capture
!pip install pybaseball pandas numpy scikit-learn matplotlib seaborn requests

In [ ]:
import os, time, warnings, requests, calendar
from datetime import date, timedelta, datetime, timezone
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pybaseball as pb
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import mean_absolute_error

pb.cache.enable()
warnings.filterwarnings('ignore')

if 'CACHE_DIR' not in dir():
    CACHE_DIR = 'cache'
    os.makedirs(CACHE_DIR, exist_ok=True)

TEAM_MAP = {
    'Arizona Diamondbacks':'AZ','Atlanta Braves':'ATL',
    'Baltimore Orioles':'BAL','Boston Red Sox':'BOS',
    'Chicago Cubs':'CHC','Chicago White Sox':'CWS',
    'Cincinnati Reds':'CIN','Cleveland Guardians':'CLE',
    'Colorado Rockies':'COL','Detroit Tigers':'DET',
    'Houston Astros':'HOU','Kansas City Royals':'KC',
    'Los Angeles Angels':'LAA','Los Angeles Dodgers':'LAD',
    'Miami Marlins':'MIA','Milwaukee Brewers':'MIL',
    'Minnesota Twins':'MIN','New York Mets':'NYM',
    'New York Yankees':'NYY','Athletics':'ATH',
    'Philadelphia Phillies':'PHI','Pittsburgh Pirates':'PIT',
    'San Diego Padres':'SD','San Francisco Giants':'SF',
    'Seattle Mariners':'SEA','St. Louis Cardinals':'STL',
    'Tampa Bay Rays':'TB','Texas Rangers':'TEX',
    'Toronto Blue Jays':'TOR','Washington Nationals':'WSH',
}
PARK_FACTORS = {
    'COL':115,'CIN':110,'BOS':108,'PHI':107,'TEX':106,'BAL':104,'NYY':103,
    'CHC':103,'MIL':102,'TOR':101,'ATL':100,'HOU':100,'LAD':100,'WSH':100,
    'CLE':100,'DET':99,'MIN':99,'STL':99,'AZ':99,'MIA':98,'NYM':98,
    'ATH':98,'PIT':98,'SEA':97,'TB':97,'CWS':96,'KC':96,'LAA':96,'SF':95,'SD':94,
}
WIND_DIR_MAP = {
    'Out to CF':1.0,'Out to RF':0.8,'Out to LF':0.8,
    'In from CF':-1.0,'In from LF':-0.8,'In from RF':-0.8,
    'L to R':0.1,'R to L':0.1,'Calm':0.0,'':0.0,
}

# ── NRFI feature list ────────────────────────────────────────────────────
NRFI_FEATURES = [
    'home_roll_k_pct','home_roll_bb_pct','home_roll_kbb',
    'home_roll_woba','home_roll_hard_pct','home_prior_xfip',
    'away_roll_k_pct','away_roll_bb_pct','away_roll_kbb',
    'away_roll_woba','away_roll_hard_pct','away_prior_xfip',
    'home_leadoff_obp','away_leadoff_obp',
    'home_team_k_pct','away_team_k_pct',
    'home_team_woba','away_team_woba',
    'park_factor','weather_offense_factor','umpire_extra_strikes',
]

# ── K model feature list ─────────────────────────────────────────────────
K_FEATURES = [
    'pitcher_avg_k',       # rolling avg Ks per game (last 10 starts)
    'pitcher_k_pct',       # rolling K% (last 10 starts)
    'pitcher_k_consistency',# 1 - std/mean (how reliable is this pitcher's K count)
    'opp_k_pct',           # how often opponent team strikes out
    'opp_woba',            # opponent team wOBA
    'park_factor',         # run environment
]

# ── Season definitions ────────────────────────────────────────────────────
SEASON_BOUNDS = {
    2024: ('2024-03-20', '2024-10-01'),
    2025: ('2025-03-27', '2025-09-28'),
    2026: ('2026-03-26', '2026-09-27'),
}
SEASON_RANGES = {
    2024:[('2024-03-20','2024-04-30'),('2024-05-01','2024-05-31'),
          ('2024-06-01','2024-06-30'),('2024-07-01','2024-07-31'),
          ('2024-08-01','2024-08-31'),('2024-09-01','2024-10-01')],
    2025:[('2025-03-27','2025-04-30'),('2025-05-01','2025-05-31'),
          ('2025-06-01','2025-06-30'),('2025-07-01','2025-07-31'),
          ('2025-08-01','2025-08-31'),('2025-09-01','2025-09-28')],
    2026:[('2026-03-26','2026-04-30'),('2026-05-01','2026-05-31'),
          ('2026-06-01','2026-06-30'),('2026-07-01','2026-07-31'),
          ('2026-08-01','2026-08-31'),('2026-09-01','2026-09-27')],
}
SEASONS_LIST = list(SEASON_RANGES.keys())

# Date locked to Eastern Time (MLB schedule timezone)
# Prevents UTC rollover causing tomorrow's slate to load in evening
ET    = timezone(timedelta(hours=-4))  # EDT (Mar-Nov); change to -5 Nov-Mar
TODAY = datetime.now(ET).strftime('%Y-%m-%d')
print(f'Setup complete. Predicting for: {TODAY}')
print(f'Cache: {CACHE_DIR}')
print(f'NRFI features: {len(NRFI_FEATURES)} | K features: {len(K_FEATURES)}')

## 2 · Load Umpire Data

**UmpScorecards** (career zone stats) + **MLB API** historical umpire assignments.
Both cached locally. UmpScorecards refreshes weekly.

- `extra_calls > 0` = bigger zone = suppresses walks = NRFI edge
- `extra_calls < 0` = tighter zone = more walks = YRFI edge
- Range across MLB umpires: roughly -1.5 to +1.5 extra strikes per game

In [ ]:
def load_umpscorecards():
    cache_path = f'{CACHE_DIR}/umpscorecards.csv'
    week_old   = 7 * 24 * 3600

    # Always define the fallback columns so the rest of the notebook
    # never hits a KeyError regardless of what the API returns
    EMPTY = pd.DataFrame(columns=[
        'name','extra_calls','correct_calls','total_games','extra_calls_z'
    ])

    if os.path.exists(cache_path):
        age = time.time() - os.path.getmtime(cache_path)
        if age < week_old:
            df = pd.read_csv(cache_path)
            # Ensure all expected columns exist even in old cache files
            for col in ['name','extra_calls','correct_calls','total_games','extra_calls_z']:
                if col not in df.columns:
                    df[col] = 0
            print(f'  UmpScorecards: {len(df)} umpires from cache')
            return df

    print('  Fetching UmpScorecards...')
    endpoints = [
        'https://umpscorecards.com/site_files/data/all_umpires_data.json',
        'https://umpscorecards.com/api/umpires',
    ]
    data = None
    for url in endpoints:
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                data = r.json()
                break
        except Exception as e:
            print(f'    {url}: {e}')

    if data is None:
        print('  UmpScorecards unreachable — using neutral (0.0) for all umpires')
        EMPTY.to_csv(cache_path, index=False)
        return EMPTY

    # Handle list or dict response
    if isinstance(data, dict):
        records = data.get('umpires', data.get('data', list(data.values())))
    else:
        records = data

    rows = []
    for u in records:
        if not isinstance(u, dict):
            continue
        name    = (u.get('name') or u.get('umpire_name') or
                   u.get('fullName') or '').strip()
        extra   = float(u.get('extra_calls') or u.get('extra_strikes') or
                        u.get('xtra_calls') or 0)
        correct = float(u.get('correct_calls') or u.get('accuracy') or 0)
        games   = int(u.get('total_games') or u.get('games') or 0)
        if name:
            rows.append({
                'name'         : name,
                'extra_calls'  : extra,
                'correct_calls': correct,
                'total_games'  : games,
            })

    # Build DataFrame with explicit columns — never empty-column surprise
    df = pd.DataFrame(rows, columns=[
        'name','extra_calls','correct_calls','total_games'
    ]) if rows else EMPTY.copy()

    # Filter to qualified umpires
    if 'total_games' in df.columns and len(df) > 0:
        df = df[df['total_games'] >= 30].copy()

    # Z-score normalization
    if len(df) > 1:
        mu    = df['extra_calls'].mean()
        sigma = df['extra_calls'].std()
        df['extra_calls_z'] = (df['extra_calls'] - mu) / (sigma + 1e-6)
    else:
        df['extra_calls_z'] = 0.0

    df.to_csv(cache_path, index=False)

    if len(df) > 0:
        print(f'  UmpScorecards: {len(df)} umpires')
        print(f'  extra_calls range: '
              f'{df["extra_calls"].min():.2f} to {df["extra_calls"].max():.2f}')
        print('  Most NRFI-friendly (biggest zone):')
        for _, r in df.nlargest(3,'extra_calls').iterrows():
            print(f'    {r["name"]:<25} {r["extra_calls"]:+.2f}')
        print('  Most YRFI-friendly (tightest zone):')
        for _, r in df.nsmallest(3,'extra_calls').iterrows():
            print(f'    {r["name"]:<25} {r["extra_calls"]:+.2f}')
    else:
        print('  No umpire data returned — neutral values (0.0) used for all games')

    return df

print('Loading UmpScorecards...')
ump_stats = load_umpscorecards()
UMP_MAP   = dict(zip(ump_stats['name'], ump_stats['extra_calls']))   if len(ump_stats)>0 else {}
UMP_MAP_Z = dict(zip(ump_stats['name'], ump_stats['extra_calls_z'])) if len(ump_stats)>0 else {}
print(f'  UMP_MAP populated: {len(UMP_MAP)} umpires')

In [ ]:
def fetch_historical_umpires(seasons):
    """
    Fetches home plate umpire for every game across given seasons
    via MLB Stats API officials hydration.
    Cached per season — only fetches once.

    Returns: DataFrame with columns [game_pk, umpire_name, umpire_id]
    """
    all_games = []
    month_ranges = {
        2022:[('2022-04-07','2022-04-30'),('2022-05-01','2022-05-31'),
              ('2022-06-01','2022-06-30'),('2022-07-01','2022-07-31'),
              ('2022-08-01','2022-08-31'),('2022-09-01','2022-10-05')],
        2023:[('2023-03-30','2023-04-30'),('2023-05-01','2023-05-31'),
              ('2023-06-01','2023-06-30'),('2023-07-01','2023-07-31'),
              ('2023-08-01','2023-08-31'),('2023-09-01','2023-10-01')],
        2024:[('2024-03-20','2024-04-30'),('2024-05-01','2024-05-31'),
              ('2024-06-01','2024-06-30'),('2024-07-01','2024-07-31'),
              ('2024-08-01','2024-08-31'),('2024-09-01','2024-10-01')],
        2025:[('2025-03-27','2025-04-30'),('2025-05-01','2025-05-31'),
              ('2025-06-01','2025-06-30'),('2025-07-01','2025-07-31'),
              ('2025-08-01','2025-08-31'),('2025-09-01','2025-09-28')],
    }

    for yr in seasons:
        cache_path = f'{CACHE_DIR}/umpires_{yr}.csv'
        if os.path.exists(cache_path):
            df = pd.read_csv(cache_path)
            print(f'  {yr}: {len(df):,} games loaded from cache')
            all_games.append(df)
            continue

        print(f'  {yr}: fetching umpire assignments from MLB API...')
        yr_games = []
        ranges = month_ranges.get(yr, [])

        for start, end in ranges:
            url = (
                f'https://statsapi.mlb.com/api/v1/schedule'
                f'?sportId=1&startDate={start}&endDate={end}'
                f'&hydrate=officials&fields=dates,games,gamePk,officials,official,fullName,id,officialType'
            )
            try:
                r = requests.get(url, timeout=15)
                data = r.json()
                for date_block in data.get('dates', []):
                    for game in date_block.get('games', []):
                        gpk = game.get('gamePk')
                        for official in game.get('officials', []):
                            if official.get('officialType') == 'Home Plate':
                                yr_games.append({
                                    'game_pk'    : gpk,
                                    'umpire_name': official['official']['fullName'],
                                    'umpire_id'  : official['official']['id'],
                                })
                time.sleep(0.5)  # polite pause
            except Exception as e:
                print(f'    Warning {start}-{end}: {e}')

        if yr_games:
            df = pd.DataFrame(yr_games).drop_duplicates('game_pk')
            df.to_csv(cache_path, index=False)
            print(f'  {yr}: {len(df):,} games cached')
            all_games.append(df)
        else:
            print(f'  {yr}: no umpire data retrieved')

    return pd.concat(all_games, ignore_index=True) if all_games else pd.DataFrame(
        columns=['game_pk','umpire_name','umpire_id'])

print('Fetching historical umpire assignments...')
SEASONS_LIST = list(SEASON_RANGES.keys())
ump_history = fetch_historical_umpires(SEASONS_LIST)

# Attach extra_calls to each historical game
ump_history['umpire_extra_strikes'] = (
    ump_history['umpire_name'].map(UMP_MAP).fillna(0.0)
)
print(f'\nUmpire coverage: {ump_history["umpire_name"].notna().mean():.1%} of games have known umpire')
if len(ump_history) > 0 and len(UMP_MAP) > 0:
    covered = ump_history['umpire_extra_strikes'].ne(0.0).mean()
    print(f'Umpire in UmpScorecards: {covered:.1%} of games')
    print(f'extra_strikes range in training: '
          f'{ump_history["umpire_extra_strikes"].min():.2f} to '
          f'{ump_history["umpire_extra_strikes"].max():.2f}')

## 3 · Fetch Today's Slate

Pulls probable pitchers **and home plate umpire** in one MLB API call.

In [ ]:
def fetch_todays_slate(date_str=TODAY):
    """
    Fetches probable pitchers AND home plate umpire for today's games.
    Both come from the MLB Stats API in a single request.
    MLBAM IDs for pitchers match Statcast directly.
    Umpire name matches UmpScorecards via string match.
    """
    url = (
        f'https://statsapi.mlb.com/api/v1/schedule'
        f'?sportId=1&date={date_str}'
        f'&hydrate=probablePitcher,team,officials'
    )
    try:
        r = requests.get(url, timeout=10)
        data = r.json()
    except Exception as e:
        print(f'API error: {e}')
        return pd.DataFrame()

    games = []
    for date_block in data.get('dates', []):
        for g in date_block.get('games', []):
            home = g['teams']['home']
            away = g['teams']['away']
            home_abbr = TEAM_MAP.get(home['team']['name'], home['team']['name'][:3].upper())
            away_abbr = TEAM_MAP.get(away['team']['name'], away['team']['name'][:3].upper())
            hp = home.get('probablePitcher', {})
            ap = away.get('probablePitcher', {})

            # Extract home plate umpire from officials list
            umpire_name = 'TBD'
            umpire_id   = None
            for official in g.get('officials', []):
                if official.get('officialType') == 'Home Plate':
                    umpire_name = official['official']['fullName']
                    umpire_id   = official['official']['id']
                    break

            # Get umpire zone stats
            ump_extra = UMP_MAP.get(umpire_name, 0.0)
            ump_games = ump_stats[ump_stats['name']==umpire_name]['total_games'].values
            ump_games = int(ump_games[0]) if len(ump_games) > 0 else 0

            games.append({
                'game_pk'          : g['gamePk'],
                'home_team'        : home_abbr,
                'away_team'        : away_abbr,
                'home_pitcher_id'  : hp.get('id'),
                'home_pitcher'     : hp.get('fullName', 'TBD'),
                'away_pitcher_id'  : ap.get('id'),
                'away_pitcher'     : ap.get('fullName', 'TBD'),
                'umpire_name'      : umpire_name,
                'umpire_id'        : umpire_id,
                'umpire_extra_strikes': ump_extra,
                'umpire_games'     : ump_games,
            })
    return pd.DataFrame(games)

slate = fetch_todays_slate()

if len(slate) == 0:
    print('No games found. Check date or add overrides below.')
else:
    print(f"Today's slate ({TODAY}): {len(slate)} games\n")
    for _, g in slate.iterrows():
        ump_tag = ''
        if g['umpire_name'] != 'TBD':
            direction = 'BIG ZONE' if g['umpire_extra_strikes'] > 0.5 else \
                        'TIGHT ZONE' if g['umpire_extra_strikes'] < -0.5 else 'avg zone'
            ump_tag = f" | UMP: {g['umpire_name']} ({g['umpire_extra_strikes']:+.2f} extra K, {direction})"
        print(f"  {g['away_team']} @ {g['home_team']}{ump_tag}")
        print(f"    Home SP: {g['home_pitcher']}  Away SP: {g['away_pitcher']}")

## 4 · Overrides (TBD Pitchers / Weather)

Only edit if a pitcher shows TBD or you want to add weather.

In [ ]:
# ── EDIT THIS CELL EACH DAY ──────────────────────────────────────────────────

# Override TBD pitchers. Format: game index (0-based) → pitcher MLBAM ID
# Example: game 0 home pitcher is TBD, set to Gerrit Cole (543037)
PITCHER_OVERRIDES = {
    # (game_index, 'home'/'away'): mlbam_id
    # (0, 'home'): 543037,
    # (2, 'away'): 605483,
}

# Weather per game. Format: game index → dict
# wind_dir options: 'Out to CF','In from CF','Out to LF','Out to RF','L to R','R to L','Calm'
# Domes / retractable roofs: set wind_speed=0, wind_dir='Calm', temperature=72
WEATHER_OVERRIDES = {
    # (0): {'wind_speed': 12, 'wind_dir': 'Out to CF', 'temperature': 68},
    # (1): {'wind_speed': 0,  'wind_dir': 'Calm',      'temperature': 72},  # dome
}

# Apply pitcher overrides to slate
for (idx, side), pid in PITCHER_OVERRIDES.items():
    if idx < len(slate):
        col = f'{side}_pitcher_id'
        slate.loc[idx, col] = pid
        print(f'Override: game {idx} {side} pitcher → MLBAM {pid}')

# Drop games where either starter is still TBD
slate_ready = slate.dropna(subset=['home_pitcher_id','away_pitcher_id']).copy()
tbd_count = len(slate) - len(slate_ready)
print(f'Games with both starters confirmed: {len(slate_ready)}/{len(slate)}')
if tbd_count > 0:
    print(f'Skipping {tbd_count} game(s) with TBD starters — add overrides above')

## 5 · Load Statcast + Train Model

Trains on full 2022-2024 dataset with umpire feature included.
Uncomment 2025 block when ready.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# AUTO-UPDATING DATA LOADER
# Detects complete months, downloads anything new, trains on freshest data.
# SEASON_BOUNDS and SEASON_RANGES are defined in the imports cell above.
# ═══════════════════════════════════════════════════════════════════════

def get_complete_months(season_start, season_end, today, buffer_days=3):
    start  = datetime.strptime(season_start, '%Y-%m-%d').date()
    end    = datetime.strptime(season_end,   '%Y-%m-%d').date()
    cutoff = today - timedelta(days=buffer_days)
    ranges = []
    current = start
    while current <= end:
        last_day = date(current.year, current.month,
                        calendar.monthrange(current.year, current.month)[1])
        month_end   = min(last_day, end)
        month_start = max(current, start)
        if month_end <= cutoff:
            ranges.append((month_start.strftime('%Y-%m-%d'),
                           month_end.strftime('%Y-%m-%d')))
        current = date(
            current.year + (1 if current.month == 12 else 0),
            1 if current.month == 12 else current.month + 1, 1)
    return ranges

def load_month(year, month_start, month_end):
    month_num  = month_start[5:7]
    cache_path = f'{CACHE_DIR}/statcast_{year}_{month_num}.csv'
    if os.path.exists(cache_path):
        df = pd.read_csv(cache_path, low_memory=False)
        df['game_date'] = pd.to_datetime(df['game_date'])
        return df, False
    print(f'    Downloading {year}-{month_num} ({month_start} to {month_end})...')
    try:
        df = pb.statcast(start_dt=month_start, end_dt=month_end)
        df['game_date'] = pd.to_datetime(df['game_date'])
        KEEP_COLS = [
            'game_pk','game_date','home_team','away_team',
            'pitcher','batter','inning','inning_topbot',
            'events','woba_value','launch_speed',
            'bat_score','post_bat_score',
            'at_bat_number','pitch_number',
            'wind_speed','wind_dir','temperature',
        ]
        df = df[[c for c in KEEP_COLS if c in df.columns]]
        df.to_csv(cache_path, index=False)
        time.sleep(3)
        return df, True
    except Exception as e:
        print(f'    Warning: {e}')
        return pd.DataFrame(), False

def load_all_seasons(season_bounds, today):
    all_chunks = []
    new_months = 0
    for year, (s_start, s_end) in sorted(season_bounds.items()):
        months = get_complete_months(s_start, s_end, today)
        if not months:
            print(f'  {year}: no complete months yet')
            continue
        yr_chunks = []
        cached_n = 0; new_n = 0
        for m_start, m_end in months:
            chunk, is_new = load_month(year, m_start, m_end)
            if len(chunk) > 0:
                yr_chunks.append(chunk)
                if is_new: new_n    += 1
                else:      cached_n += 1
        if yr_chunks:
            yr_df  = pd.concat(yr_chunks, ignore_index=True)
            games  = yr_df['game_pk'].nunique()
            new_months += new_n
            all_chunks.append(yr_df)
            status = f'{cached_n} cached'
            if new_n > 0: status += f' + {new_n} NEW'
            print(f'  {year}: {games:,} games ({len(months)} months — {status})')
    if new_months > 0:
        print(f'\n  {new_months} new month(s) downloaded and saved to Drive')
    return pd.concat(all_chunks, ignore_index=True) if all_chunks else pd.DataFrame()

print(f'Loading Statcast (auto-updating)...')
sc = load_all_seasons(SEASON_BOUNDS, date.today())

if len(sc) == 0:
    raise RuntimeError('No data loaded — check cache directory and network')

sc['game_date'] = pd.to_datetime(sc['game_date'])
sc = sc.sort_values('game_date').reset_index(drop=True)
print(f'\nTotal: {sc["game_pk"].nunique():,} games | '
      f'{sc["game_date"].min().date()} to {sc["game_date"].max().date()}')

# ── Rolling pitcher stats ─────────────────────────────────────────────────
print('\nBuilding rolling pitcher stats...')
pa = sc[sc['events'].notna()].copy()
pa['hard_hit'] = (
    pd.to_numeric(pa.get('launch_speed', np.nan), errors='coerce') >= 95
).astype(float)
gstats = (
    pa.groupby(['pitcher','game_pk','game_date'])
    .agg(pa=('events','count'),
         k =('events', lambda x:(x=='strikeout').sum()),
         bb=('events', lambda x:(x=='walk').sum()),
         woba=('woba_value','mean'),
         hard_hit=('hard_hit','mean'))
    .reset_index().sort_values(['pitcher','game_date'])
)
N = 10
def rs(s): return s.shift(1).rolling(N, min_periods=4).sum()
def rm(s): return s.shift(1).rolling(N, min_periods=4).mean()
g = gstats.groupby('pitcher')
gstats['roll_pa']   = g['pa'].transform(rs)
gstats['roll_k']    = g['k'].transform(rs)
gstats['roll_bb']   = g['bb'].transform(rs)
gstats['roll_woba'] = g['woba'].transform(rm)
gstats['roll_hh']   = g['hard_hit'].transform(rm)
gstats['roll_k_pct']    = gstats['roll_k']  / gstats['roll_pa'].replace(0, np.nan)
gstats['roll_bb_pct']   = gstats['roll_bb'] / gstats['roll_pa'].replace(0, np.nan)
gstats['roll_kbb']      = gstats['roll_k_pct'] - gstats['roll_bb_pct']
gstats['roll_hard_pct'] = gstats['roll_hh'].fillna(0.35)
rolling = gstats[gstats['roll_pa'] >= 20][[
    'pitcher','game_pk','game_date',
    'roll_k_pct','roll_bb_pct','roll_kbb','roll_woba','roll_hard_pct'
]].copy()
print(f'  {rolling["pitcher"].nunique():,} pitchers in rolling stats')

# ── Team batting stats ────────────────────────────────────────────────────
pa2 = sc[sc['events'].notna()].copy()
pa2['batting_team'] = np.where(
    pa2['inning_topbot']=='Top', pa2['away_team'], pa2['home_team'])
pa2['season'] = pa2['game_date'].dt.year
team_stats = pa2.groupby(['batting_team','season']).agg(
    team_woba  =('woba_value','mean'),
    team_k_pct =('events', lambda x:(x=='strikeout').sum()/len(x))
).reset_index()

# ── Leadoff OBP ───────────────────────────────────────────────────────────
ON_BASE = {'single','double','triple','home_run','walk','hit_by_pitch'}
pa2['on_base'] = pa2['events'].isin(ON_BASE).astype(int)
batter_pa = (
    pa2.groupby(['batter','game_pk','game_date'])
    .agg(pa_c=('events','count'), ob_c=('on_base','sum'))
    .reset_index().sort_values(['batter','game_date'])
)
batter_pa['trail_pa'] = batter_pa.groupby('batter')['pa_c'].transform(
    lambda x: x.shift(1).rolling(50, min_periods=10).sum())
batter_pa['trail_ob'] = batter_pa.groupby('batter')['ob_c'].transform(
    lambda x: x.shift(1).rolling(50, min_periods=10).sum())
batter_pa['trail_obp'] = (
    batter_pa['trail_ob'] / batter_pa['trail_pa'].replace(0, np.nan))
latest_obp = (
    batter_pa.dropna(subset=['trail_obp'])
    .sort_values('game_date')
    .groupby('batter')['trail_obp'].last()
    .reset_index().rename(columns={'trail_obp':'obp'})
)
inn1 = sc[sc['inning']==1].copy()
inn1['batting_team'] = np.where(
    inn1['inning_topbot']=='Top', inn1['away_team'], inn1['home_team'])
inn1s = inn1.sort_values(
    ['game_pk','inning_topbot','at_bat_number','pitch_number'])
leadoff_id = inn1s.groupby(
    ['game_pk','batting_team'])['batter'].first().reset_index()
leadoff_id['game_date'] = leadoff_id['game_pk'].map(
    sc.groupby('game_pk')['game_date'].first())
recent_cut = sc['game_date'].max() - pd.Timedelta(days=45)
typical_lo = (
    leadoff_id[leadoff_id['game_date'] >= recent_cut]
    .groupby('batting_team')['batter']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index().rename(columns={'batter':'leadoff_batter_id'})
)
typical_lo = typical_lo.merge(
    latest_obp, left_on='leadoff_batter_id', right_on='batter', how='left')
typical_lo['obp'] = typical_lo['obp'].fillna(0.320)
LEADOFF_OBP = dict(zip(typical_lo['batting_team'], typical_lo['obp']))

# ── FanGraphs xFIP (prior complete season, auto-detected) ─────────────────
print('\nLoading FanGraphs xFIP...')
XFIP_MAP  = {}
xfip_year = date.today().year - 1
try:
    fg_cache = f'{CACHE_DIR}/fg_pitching_{xfip_year}.csv'
    fg = (pd.read_csv(fg_cache) if os.path.exists(fg_cache)
          else pb.pitching_stats(xfip_year, xfip_year, qual=30))
    if not os.path.exists(fg_cache): fg.to_csv(fg_cache, index=False)
    xfip_col = next(
        (c for c in fg.columns if 'xfip' in c.lower() and '-' not in c.lower()), None)
    idfg_col = next(
        (c for c in fg.columns if 'idfg' in c.lower() or c=='IDfg'), None)
    if xfip_col and idfg_col:
        fg_sub = fg[[idfg_col,xfip_col]].rename(
            columns={idfg_col:'fg_id', xfip_col:'xfip'})
        fg_sub['fg_id'] = fg_sub['fg_id'].astype(int)
        lookup = pb.playerid_reverse_lookup(
            fg_sub['fg_id'].tolist(), key_type='fangraphs')
        lookup = lookup[['key_fangraphs','key_mlbam']].rename(
            columns={'key_fangraphs':'fg_id','key_mlbam':'pitcher'})
        fg_sub = fg_sub.merge(lookup, on='fg_id', how='left')
        XFIP_MAP = dict(zip(fg_sub['pitcher'].astype('Int64'), fg_sub['xfip']))
        print(f'  xFIP ({xfip_year}): {len(XFIP_MAP)} pitchers')
except Exception as e:
    print(f'  FanGraphs error: {e} — using league average 4.25')

# ── Train model ───────────────────────────────────────────────────────────
print('\nTraining model...')
inn1_sc = sc[sc['inning']==1].copy()
meta = sc.groupby('game_pk').agg(
    game_date=('game_date','first'),
    home_team=('home_team','first'),
    away_team=('away_team','first'),
).reset_index()
def half_runs(g):
    return (g['post_bat_score']-g['bat_score']).clip(lower=0).sum()
tr = (inn1_sc[inn1_sc['inning_topbot']=='Top']
      .groupby('game_pk').apply(half_runs).reset_index(name='top_r'))
br = (inn1_sc[inn1_sc['inning_topbot']=='Bot']
      .groupby('game_pk').apply(half_runs).reset_index(name='bot_r'))
meta = meta.merge(tr,on='game_pk',how='left').merge(br,on='game_pk',how='left')
meta[['top_r','bot_r']] = meta[['top_r','bot_r']].fillna(0)
meta['YRFI'] = ((meta['top_r']>0)|(meta['bot_r']>0)).astype(int)
inn1s2 = inn1_sc.sort_values(['game_pk','inning_topbot','pitch_number'])
hp = (inn1s2[inn1s2['inning_topbot']=='Top']
      .groupby('game_pk')['pitcher'].first().reset_index()
      .rename(columns={'pitcher':'hsp'}))
ap = (inn1s2[inn1s2['inning_topbot']=='Bot']
      .groupby('game_pk')['pitcher'].first().reset_index()
      .rename(columns={'pitcher':'asp'}))
meta = meta.merge(hp,on='game_pk',how='left').merge(ap,on='game_pk',how='left')
meta['game_date']  = pd.to_datetime(meta['game_date'])
meta['season']     = meta['game_date'].dt.year
meta['park_factor']= meta['home_team'].map(PARK_FACTORS).fillna(100)/100.0
meta['weather_offense_factor'] = 0.0
meta = meta.merge(
    ump_history[['game_pk','umpire_extra_strikes']],
    on='game_pk', how='left')
meta['umpire_extra_strikes'] = meta['umpire_extra_strikes'].fillna(0.0)
roll_h = rolling.rename(columns={
    'roll_k_pct':'h_kp','roll_bb_pct':'h_bbp','roll_kbb':'h_kbb',
    'roll_woba':'h_woba','roll_hard_pct':'h_hard'})
roll_a = rolling.rename(columns={
    'roll_k_pct':'a_kp','roll_bb_pct':'a_bbp','roll_kbb':'a_kbb',
    'roll_woba':'a_woba','roll_hard_pct':'a_hard'})
train_df = meta.merge(
    roll_h[['pitcher','game_pk','h_kp','h_bbp','h_kbb','h_woba','h_hard']],
    left_on=['hsp','game_pk'],right_on=['pitcher','game_pk'],
    how='inner').drop(columns=['pitcher'])
train_df = train_df.merge(
    roll_a[['pitcher','game_pk','a_kp','a_bbp','a_kbb','a_woba','a_hard']],
    left_on=['asp','game_pk'],right_on=['pitcher','game_pk'],
    how='inner').drop(columns=['pitcher'])
train_df['home_prior_xfip'] = train_df['hsp'].map(XFIP_MAP).fillna(4.25)
train_df['away_prior_xfip'] = train_df['asp'].map(XFIP_MAP).fillna(4.25)
ts_h = team_stats.rename(columns={
    'batting_team':'t','team_woba':'h_twoba','team_k_pct':'h_tkp'})
ts_a = team_stats.rename(columns={
    'batting_team':'t','team_woba':'a_twoba','team_k_pct':'a_tkp'})
train_df = train_df.merge(
    ts_h,left_on=['home_team','season'],right_on=['t','season'],
    how='left').drop(columns=['t'])
train_df = train_df.merge(
    ts_a,left_on=['away_team','season'],right_on=['t','season'],
    how='left').drop(columns=['t'])
train_df['home_leadoff_obp'] = train_df['home_team'].map(LEADOFF_OBP).fillna(0.320)
train_df['away_leadoff_obp'] = train_df['away_team'].map(LEADOFF_OBP).fillna(0.320)
rename_map = {
    'h_kp':'home_roll_k_pct','h_bbp':'home_roll_bb_pct','h_kbb':'home_roll_kbb',
    'h_woba':'home_roll_woba','h_hard':'home_roll_hard_pct',
    'a_kp':'away_roll_k_pct','a_bbp':'away_roll_bb_pct','a_kbb':'away_roll_kbb',
    'a_woba':'away_roll_woba','a_hard':'away_roll_hard_pct',
    'h_twoba':'home_team_woba','h_tkp':'home_team_k_pct',
    'a_twoba':'away_team_woba','a_tkp':'away_team_k_pct',
}
train_df    = train_df.rename(columns=rename_map)
train_clean = train_df.dropna(subset=NRFI_FEATURES+['YRFI'])
scaler = StandardScaler()
X = scaler.fit_transform(train_clean[NRFI_FEATURES])
y = train_clean['YRFI'].values
base  = LogisticRegression(C=0.5,max_iter=2000,class_weight='balanced',random_state=42)
model = CalibratedClassifierCV(base, method='isotonic', cv=5)
model.fit(X, y)
seasons_used = sorted(train_clean['season'].unique().tolist())
print(f'Model trained on {len(train_clean):,} games')
print(f'Seasons: {seasons_used} | YRFI rate: {y.mean():.1%}')

## 6 · Build Features

In [ ]:
def get_pitcher_rolling(pitcher_id, rolling_df):
    p = rolling_df[rolling_df['pitcher']==pitcher_id].sort_values('game_date')
    if len(p) == 0:
        return {'roll_k_pct':0.220,'roll_bb_pct':0.085,'roll_kbb':0.135,
                'roll_woba':0.315,'roll_hard_pct':0.370,'found':False}
    last = p.iloc[-1]
    return {'roll_k_pct':last['roll_k_pct'],'roll_bb_pct':last['roll_bb_pct'],
            'roll_kbb':last['roll_kbb'],'roll_woba':last['roll_woba'],
            'roll_hard_pct':last['roll_hard_pct'],'found':True,
            'last_start':str(last['game_date'].date()),'n_starts':len(p)}

def build_game_features(game, weather_overrides, idx):
    home = game['home_team']; away = game['away_team']
    hpid = int(game['home_pitcher_id']); apid = int(game['away_pitcher_id'])
    hp = get_pitcher_rolling(hpid, rolling)
    ap = get_pitcher_rolling(apid, rolling)

    wx = weather_overrides.get(idx, {})
    wind_speed  = wx.get('wind_speed',  5.0)
    wind_dir    = wx.get('wind_dir',    'Calm')
    temperature = wx.get('temperature', 72.0)
    wind_factor = WIND_DIR_MAP.get(wind_dir, 0.0)
    weather_factor = wind_factor*(wind_speed/10.0)+(temperature-72)/30.0

    latest_season = team_stats['season'].max()
    home_ts = team_stats[(team_stats['batting_team']==home)&(team_stats['season']==latest_season)]
    away_ts = team_stats[(team_stats['batting_team']==away)&(team_stats['season']==latest_season)]
    h_woba = home_ts['team_woba'].values[0] if len(home_ts)>0 else 0.315
    h_kpct = home_ts['team_k_pct'].values[0] if len(home_ts)>0 else 0.220
    a_woba = away_ts['team_woba'].values[0] if len(away_ts)>0 else 0.315
    a_kpct = away_ts['team_k_pct'].values[0] if len(away_ts)>0 else 0.220

    # Umpire zone feature — from slate (already looked up)
    ump_extra = float(game.get('umpire_extra_strikes', 0.0))

    return {
        'home_roll_k_pct'       : hp['roll_k_pct'],
        'home_roll_bb_pct'      : hp['roll_bb_pct'],
        'home_roll_kbb'         : hp['roll_kbb'],
        'home_roll_woba'        : hp['roll_woba'],
        'home_roll_hard_pct'    : hp['roll_hard_pct'],
        'home_prior_xfip'       : XFIP_MAP.get(hpid, 4.25),
        'away_roll_k_pct'       : ap['roll_k_pct'],
        'away_roll_bb_pct'      : ap['roll_bb_pct'],
        'away_roll_kbb'         : ap['roll_kbb'],
        'away_roll_woba'        : ap['roll_woba'],
        'away_roll_hard_pct'    : ap['roll_hard_pct'],
        'away_prior_xfip'       : XFIP_MAP.get(apid, 4.25),
        'home_leadoff_obp'      : LEADOFF_OBP.get(home, 0.320),
        'away_leadoff_obp'      : LEADOFF_OBP.get(away, 0.320),
        'home_team_k_pct'       : h_kpct,
        'away_team_k_pct'       : a_kpct,
        'home_team_woba'        : h_woba,
        'away_team_woba'        : a_woba,
        'park_factor'           : PARK_FACTORS.get(home, 100)/100.0,
        'weather_offense_factor': weather_factor,
        'umpire_extra_strikes'  : ump_extra,
    }, hp, ap

print('Building feature rows...')
feature_rows=[]; meta_rows=[]
for idx, game in slate_ready.iterrows():
    row, hp, ap = build_game_features(game, WEATHER_OVERRIDES, idx)
    feature_rows.append(row)
    meta_rows.append({
        'matchup'          : f"{game['away_team']} @ {game['home_team']}",
        'home_team'        : game['home_team'],
        'away_team'        : game['away_team'],
        'home_pitcher'     : game['home_pitcher'],
        'away_pitcher'     : game['away_pitcher'],
        'umpire'           : game.get('umpire_name','TBD'),
        'umpire_extra_k'   : game.get('umpire_extra_strikes', 0.0),
        'home_sp_found'    : hp.get('found', False),
        'away_sp_found'    : ap.get('found', False),
        'home_last_start'  : hp.get('last_start','unknown'),
        'away_last_start'  : ap.get('last_start','unknown'),
    })
feat_df = pd.DataFrame(feature_rows)
meta_df = pd.DataFrame(meta_rows)
print(f'Feature rows: {len(feat_df)}')
not_found = meta_df[~meta_df['home_sp_found']|~meta_df['away_sp_found']]
if len(not_found)>0:
    print('WARNING — pitchers not in cache (using league avg):')
    for _,r in not_found.iterrows():
        if not r['home_sp_found']: print(f'  {r["home_pitcher"]} ({r["home_team"]})')
        if not r['away_sp_found']: print(f'  {r["away_pitcher"]} ({r["away_team"]})')
tbd_ump = meta_df[meta_df['umpire']=='TBD']
if len(tbd_ump)>0:
    print(f'Umpire TBD for {len(tbd_ump)} game(s) — using neutral zone (0.0)')

## 7 · Run Predictions

In [ ]:
X_today = scaler.transform(feat_df[NRFI_FEATURES].fillna(feat_df[NRFI_FEATURES].mean()))
yrfi_prob = model.predict_proba(X_today)[:,1]
nrfi_prob = 1 - yrfi_prob

results = meta_df.copy()
results['yrfi_prob'] = yrfi_prob.round(4)
results['nrfi_prob'] = nrfi_prob.round(4)
results['edge']      = (np.abs(yrfi_prob - 0.5)*100).round(1)

def label(p):
    if p < 0.43:   return 'NRFI','STRONG'
    elif p < 0.47: return 'NRFI','LEAN'
    elif p > 0.57: return 'YRFI','STRONG'
    elif p > 0.53: return 'YRFI','LEAN'
    else:          return 'SKIP','TOSS-UP'

results[['call','confidence']] = pd.DataFrame(
    [label(p) for p in yrfi_prob], index=results.index)

results = results.sort_values('edge', ascending=False).reset_index(drop=True)
results.index += 1

print(f'Predictions for {TODAY}')
print('='*75)
for i, r in results.iterrows():
    flag = '*** BET' if r['confidence'] in ['STRONG','LEAN'] else '    skip'
    ump_tag = ''
    if r.get('umpire','TBD') != 'TBD':
        ek = r.get('umpire_extra_k',0)
        zone = 'BIG ZONE' if ek>0.5 else 'TIGHT ZONE' if ek<-0.5 else 'avg'
        ump_tag = f" | {r['umpire']} ({ek:+.1f} K, {zone})"
    print(f"  {i:>2}. {r['matchup']:<18} {r['call']:<5} {r['confidence']:<8} "
          f"edge={r['edge']:>4.1f}%  {flag}")
    print(f"       SP: {r['away_pitcher']:<22} vs {r['home_pitcher']}")
    print(f"       UMP:{ump_tag if ump_tag else ' TBD'}")

bettable = results[results['confidence'].isin(['STRONG','LEAN'])]
print(f'\nBettable: {len(bettable)} | Skip: {len(results)-len(bettable)}')
print(f'Backtest: Strong ~58.6% | Lean ~56.4% | Toss-up ~50.8%')

## 8 · Visual Card

## 7 · K Model Training

Builds strikeout prediction model from the same Statcast data.
Ridge regression predicting raw K count per start.
**67.6% O/U accuracy in backtest — this is the primary model.**

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# K MODEL TRAINING
# Ridge regression predicting raw strikeout count per starter per game.
# O/U line = median Ks in training set.
# Backtest accuracy: 67.6%
# ═══════════════════════════════════════════════════════════════════════

print('Building K model training data...')

# Per-pitcher per-game K counts from Statcast
pa_all = sc[sc['events'].notna()].copy()
game_k = (
    pa_all.groupby(['pitcher','game_pk','game_date'])
    .agg(
        k_count  = ('events', lambda x:(x=='strikeout').sum()),
        pa_faced = ('events', 'count'),
    )
    .reset_index()
)

# Season-level pitcher K aggregates (rolling already computed above)
# Build per-pitcher K stats from rolling DataFrame
pitcher_k_stats = (
    game_k.groupby('pitcher')
    .agg(
        games           = ('game_pk',  'nunique'),
        total_k         = ('k_count',  'sum'),
        total_pa        = ('pa_faced', 'sum'),
        avg_k_per_game  = ('k_count',  'mean'),
        std_k_per_game  = ('k_count',  'std'),
        median_k        = ('k_count',  'median'),
    )
    .reset_index()
)
pitcher_k_stats = pitcher_k_stats[pitcher_k_stats['games'] >= 5].copy()
pitcher_k_stats['k_pct_full']    = pitcher_k_stats['total_k'] / pitcher_k_stats['total_pa']
pitcher_k_stats['k_consistency'] = (
    1 - pitcher_k_stats['std_k_per_game'] /
    (pitcher_k_stats['avg_k_per_game'] + 0.01)
)

# Build training rows: one per starter per game
inn1_sc2 = sc[sc['inning']==1].copy()
inn1s_k  = inn1_sc2.sort_values(['game_pk','inning_topbot','pitch_number'])
home_sp  = (inn1s_k[inn1s_k['inning_topbot']=='Top']
            .groupby('game_pk')['pitcher'].first().reset_index()
            .rename(columns={'pitcher':'pitcher_id'}))
home_sp['batting_team'] = home_sp['game_pk'].map(
    sc.groupby('game_pk')['away_team'].first())
home_sp['home_team'] = home_sp['game_pk'].map(
    sc.groupby('game_pk')['home_team'].first())
home_sp['game_date'] = home_sp['game_pk'].map(
    sc.groupby('game_pk')['game_date'].first())

away_sp = (inn1s_k[inn1s_k['inning_topbot']=='Bot']
           .groupby('game_pk')['pitcher'].first().reset_index()
           .rename(columns={'pitcher':'pitcher_id'}))
away_sp['batting_team'] = away_sp['game_pk'].map(
    sc.groupby('game_pk')['home_team'].first())
away_sp['home_team'] = away_sp['game_pk'].map(
    sc.groupby('game_pk')['home_team'].first())
away_sp['game_date'] = away_sp['game_pk'].map(
    sc.groupby('game_pk')['game_date'].first())

starts = pd.concat([home_sp, away_sp], ignore_index=True)
starts['game_date'] = pd.to_datetime(starts['game_date'])
starts['season']    = starts['game_date'].dt.year

# Attach actual K count
starts = starts.merge(
    game_k[['game_pk','pitcher','k_count']],
    left_on=['game_pk','pitcher_id'], right_on=['game_pk','pitcher'],
    how='inner'
).drop(columns=['pitcher'])

# Attach pitcher K stats
starts = starts.merge(
    pitcher_k_stats[['pitcher','avg_k_per_game','k_pct_full',
                      'k_consistency','median_k','std_k_per_game']],
    left_on='pitcher_id', right_on='pitcher', how='inner'
).drop(columns=['pitcher'])

# Attach opponent team batting stats
opp_stats = team_stats.rename(columns={
    'batting_team':'bt','team_woba':'opp_woba','team_k_pct':'opp_k_pct'})
starts = starts.merge(
    opp_stats, left_on=['batting_team','season'], right_on=['bt','season'],
    how='left'
).drop(columns=['bt'])

# Park factor
starts['park_factor'] = starts['home_team'].map(PARK_FACTORS).fillna(100)/100.0

# Rename to match K_FEATURES
starts = starts.rename(columns={
    'avg_k_per_game' : 'pitcher_avg_k',
    'k_pct_full'     : 'pitcher_k_pct',
    'k_consistency'  : 'pitcher_k_consistency',
})

k_train = starts.dropna(subset=K_FEATURES+['k_count']).copy()
k_train = k_train.sort_values('game_date').reset_index(drop=True)

# Train Ridge regression on full dataset
k_scaler = StandardScaler()
Xk = k_scaler.fit_transform(k_train[K_FEATURES])
yk = k_train['k_count'].values
k_model = Ridge(alpha=2.0)
k_model.fit(Xk, yk)

# K line = median Ks in training set
K_LINE = float(k_train['k_count'].median())

# Build latest pitcher K stats for predictions
PITCHER_K_STATS = pitcher_k_stats.set_index('pitcher')[[
    'avg_k_per_game','k_pct_full','k_consistency'
]].rename(columns={
    'avg_k_per_game':'pitcher_avg_k',
    'k_pct_full'    :'pitcher_k_pct',
    'k_consistency' :'pitcher_k_consistency',
})

print(f'K model trained on {len(k_train):,} pitcher-game rows')
print(f'K line (median): {K_LINE:.1f} Ks')
print(f'Pitchers in K stats: {len(PITCHER_K_STATS):,}')

## 8 · K Predictions

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# K MODEL PREDICTIONS — one row per starting pitcher per game
# ═══════════════════════════════════════════════════════════════════════

def get_pitcher_k_features(pitcher_id, batting_team, home_team, season):
    """Build K feature row for one pitcher start."""
    # Pitcher stats
    if pitcher_id in PITCHER_K_STATS.index:
        ps = PITCHER_K_STATS.loc[pitcher_id]
        avg_k = float(ps['pitcher_avg_k'])
        k_pct = float(ps['pitcher_k_pct'])
        k_con = float(ps['pitcher_k_consistency'])
        found = True
    else:
        # League average fallback
        avg_k = 5.0; k_pct = 0.220; k_con = 0.60
        found = False

    # Opponent team stats
    opp = team_stats[
        (team_stats['batting_team']==batting_team) &
        (team_stats['season']==season)
    ]
    opp_k   = float(opp['team_k_pct'].values[0]) if len(opp)>0 else 0.220
    opp_w   = float(opp['team_woba'].values[0])   if len(opp)>0 else 0.315
    park_f  = PARK_FACTORS.get(home_team, 100) / 100.0

    return {
        'pitcher_avg_k'        : avg_k,
        'pitcher_k_pct'        : k_pct,
        'pitcher_k_consistency': k_con,
        'opp_k_pct'            : opp_k,
        'opp_woba'             : opp_w,
        'park_factor'          : park_f,
        'found'                : found,
    }

print(f'Building K predictions for {TODAY}...')
k_rows = []
k_meta = []
current_season = date.today().year

for _, game in slate_ready.iterrows():
    for role, pitcher_id, pitcher_name, batting_team in [
        ('home', int(game['home_pitcher_id']), game['home_pitcher'], game['away_team']),
        ('away', int(game['away_pitcher_id']), game['away_pitcher'], game['home_team']),
    ]:
        feats = get_pitcher_k_features(
            pitcher_id, batting_team, game['home_team'], current_season)
        k_rows.append({k:v for k,v in feats.items() if k != 'found'})
        k_meta.append({
            'matchup'     : f"{game['away_team']} @ {game['home_team']}",
            'pitcher'     : pitcher_name,
            'pitcher_id'  : pitcher_id,
            'role'        : role,
            'opp_team'    : batting_team,
            'pitcher_found': feats['found'],
        })

k_feat_df = pd.DataFrame(k_rows)
k_meta_df = pd.DataFrame(k_meta)

# Predict
Xk_today    = k_scaler.transform(k_feat_df[K_FEATURES])
predicted_k = k_model.predict(Xk_today)

k_meta_df['predicted_k'] = predicted_k.round(2)
k_meta_df['k_line']      = K_LINE
k_meta_df['edge_k']      = (predicted_k - K_LINE).round(2)
k_meta_df['abs_edge']    = np.abs(k_meta_df['edge_k'])

def k_label(edge):
    direction = 'OVER' if edge > 0 else 'UNDER'
    if abs(edge) >= 1.5:  return direction, 'STRONG'
    elif abs(edge) >= 0.75: return direction, 'LEAN'
    else:                 return direction, 'SKIP'

k_meta_df[['k_call','k_confidence']] = pd.DataFrame(
    [k_label(e) for e in k_meta_df['edge_k']], index=k_meta_df.index)

k_results = k_meta_df.sort_values('abs_edge', ascending=False).reset_index(drop=True)
k_results.index += 1

print(f'\n{"="*65}')
print(f'  STRIKEOUT PREDICTIONS — {TODAY}  (line: {K_LINE:.1f} Ks)')
print(f'  Backtest accuracy: 67.6% O/U')
print(f'{"="*65}')
for i, r in k_results.iterrows():
    flag = '*** BET' if r['k_confidence'] in ['STRONG','LEAN'] else '    skip'
    not_found = ' [league avg]' if not r['pitcher_found'] else ''
    print(f"  {i:>2}. {r['pitcher']:<25} {r['k_call']:<6} {r['k_confidence']:<8}"
          f" pred={r['predicted_k']:.1f}K  line={K_LINE:.1f}  "
          f"edge={r['edge_k']:+.2f}  {flag}{not_found}")
    print(f"       {r['matchup']}  vs {r['opp_team']}")

k_bettable = k_results[k_results['k_confidence'].isin(['STRONG','LEAN'])]
print(f'\nBettable K props: {len(k_bettable)} | Skip: {len(k_results)-len(k_bettable)}')

## 9 · Combined Chart

In [ ]:
sns.set_theme(style='darkgrid', font_scale=0.95)
fig = plt.figure(figsize=(22, max(10, max(len(k_results), len(results))*0.55 + 4)))
fig.patch.set_facecolor('#0f1117')
fig.suptitle(f'MLB Daily Predictions — {TODAY}',
             fontsize=16, fontweight='bold', color='white', y=0.99)
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.4)
BG,T,G,R,O,GR = '#1a1d27','#e0e0e0','#4CAF50','#F44336','#FF9800','#9E9E9E'

def sax(ax, title):
    ax.set_facecolor(BG)
    ax.set_title(title, fontsize=12, fontweight='bold', color=T, pad=10)
    ax.tick_params(colors=T)
    ax.xaxis.label.set_color(T)
    [s.set_edgecolor('#444') for s in ax.spines.values()]

# ── LEFT: K model predictions ─────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor(BG)
k_labels = [f"{r['pitcher'].split()[-1]}\n{r['matchup']}"
            for _, r in k_results.iterrows()]
k_colors = []
for _, r in k_results.iterrows():
    if r['k_call']=='OVER'  and r['k_confidence']=='STRONG': k_colors.append(R)
    elif r['k_call']=='OVER'  and r['k_confidence']=='LEAN':   k_colors.append('#E57373')
    elif r['k_call']=='UNDER' and r['k_confidence']=='STRONG': k_colors.append(G)
    elif r['k_call']=='UNDER' and r['k_confidence']=='LEAN':   k_colors.append('#81C784')
    else: k_colors.append(GR)

bars1 = ax1.barh(range(len(k_results)), k_results['predicted_k'].values,
                 color=k_colors, height=0.6, edgecolor='none')
ax1.axvline(K_LINE, color='white', lw=2, ls='--', alpha=0.8,
            label=f'Line: {K_LINE:.1f} Ks')
ax1.set_yticks(range(len(k_results)))
ax1.set_yticklabels(k_labels, color=T, fontsize=8)
ax1.set_xlabel('Predicted Strikeouts', color=T)
ax1.legend(facecolor=BG, labelcolor=T, fontsize=9)
for bar, (_, r) in zip(bars1, k_results.iterrows()):
    ax1.text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
             f"{r['k_call']} ({r['k_confidence']}) {r['edge_k']:+.1f}",
             va='center', color=T, fontsize=8)
sax(ax1, f'Strikeout O/U  (line {K_LINE:.1f} Ks) — 67.6% backtest')

# ── RIGHT: NRFI predictions ───────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor(BG)
nrfi_labels = [
    f"{r['matchup']}\n{r['away_pitcher'].split()[-1]} vs {r['home_pitcher'].split()[-1]}"
    for _, r in results.iterrows()
]
nrfi_colors = []
for _, r in results.iterrows():
    if r['call']=='NRFI' and r['confidence']=='STRONG':   nrfi_colors.append(G)
    elif r['call']=='NRFI' and r['confidence']=='LEAN':   nrfi_colors.append('#81C784')
    elif r['call']=='YRFI' and r['confidence']=='STRONG': nrfi_colors.append(R)
    elif r['call']=='YRFI' and r['confidence']=='LEAN':   nrfi_colors.append('#E57373')
    else: nrfi_colors.append(GR)

bars2 = ax2.barh(range(len(results)), results['yrfi_prob'].values,
                 color=nrfi_colors, height=0.6, edgecolor='none')
ax2.axvline(0.50, color='white', lw=2, ls='--', alpha=0.8, label='50% line')
ax2.axvline(0.43, color=G, lw=1, ls=':', alpha=0.5, label='NRFI zone')
ax2.axvline(0.57, color=R, lw=1, ls=':', alpha=0.5, label='YRFI zone')
ax2.set_yticks(range(len(results)))
ax2.set_yticklabels(nrfi_labels, color=T, fontsize=8)
ax2.set_xlabel('YRFI Probability', color=T)
ax2.set_xlim(0.2, 0.9)
ax2.legend(facecolor=BG, labelcolor=T, fontsize=9)
for bar, (_, r) in zip(bars2, results.iterrows()):
    ax2.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
             f"{r['call']} ({r['confidence']})",
             va='center', color=T, fontsize=8)
sax(ax2, 'NRFI/YRFI  (bet STRONG/LEAN only) — 56% backtest')

plt.tight_layout(rect=[0, 0, 1, 0.97])
fname = f'mlb_predictions_{TODAY}.png'
plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Chart saved: {fname}')

## 9 · Save & Download

## 10 · Save & Download

In [ ]:
# Save K predictions
k_out = k_results[['matchup','pitcher','role','opp_team',
                    'k_call','k_confidence','predicted_k','k_line','edge_k']]
k_out.to_csv(f'k_predictions_{TODAY}.csv', index=False)

# Save NRFI predictions
nrfi_out_cols = [c for c in ['matchup','home_pitcher','away_pitcher',
    'umpire','umpire_extra_k','call','confidence','nrfi_prob','yrfi_prob','edge']
    if c in results.columns]
results[nrfi_out_cols].to_csv(f'nrfi_predictions_{TODAY}.csv', index=False)

print(f'Saved: k_predictions_{TODAY}.csv')
print(f'Saved: nrfi_predictions_{TODAY}.csv')

try:
    from google.colab import files
    files.download(f'k_predictions_{TODAY}.csv')
    files.download(f'nrfi_predictions_{TODAY}.csv')
    files.download(f'mlb_predictions_{TODAY}.png')
    print('Downloads triggered.')
except ImportError:
    print('(Not in Colab - files saved locally)')

# ── Final summary ─────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print(f'  MLB PICKS — {TODAY}')
print(f'{"="*60}')
print(f'\n  STRIKEOUTS (67.6% backtest — PRIMARY)')
print(f'  Line: {K_LINE:.1f} Ks')
for _, r in k_results[k_results['k_confidence'].isin(['STRONG','LEAN'])].iterrows():
    print(f"  {r['k_call']:<6} {r['k_confidence']:<8} {r['pitcher']:<25}"
          f" pred={r['predicted_k']:.1f}K  edge={r['edge_k']:+.2f}")
    print(f"         {r['matchup']}")

print(f'\n  NRFI/YRFI (56% backtest — FILTER)')
for _, r in results[results['confidence'].isin(['STRONG','LEAN'])].iterrows():
    ump = f" | {r.get('umpire','TBD')}" if r.get('umpire','TBD')!='TBD' else ''
    print(f"  {r['call']:<5} {r['confidence']:<8} {r['matchup']}{ump}")
    print(f"         NRFI {r['nrfi_prob']*100:.1f}% | YRFI {r['yrfi_prob']*100:.1f}%")

print(f'\n{"="*60}')